# TF-IDF — Term Frequency × Inverse Document Frequency

**Goal:** Score how important a word is to a document, relative to a collection of documents (corpus).

---

## Theory

### Term Frequency (TF)
How often does a word appear in *this* document?

```
TF(word, doc) = count of word in doc / total words in doc
```

### Inverse Document Frequency (IDF)
How rare is the word across *all* documents?

```
IDF(word) = log10( total_docs / docs_containing_word )
```

- A word in **every** document → IDF = log10(1) = **0** (not informative)
- A word in **one** document → IDF = log10(n) → **high** (distinctive)

### TF-IDF
```
TF-IDF = TF × IDF
```

High score = word is **frequent in this doc** AND **rare across others** → highly relevant

---

### Example
Corpus: 2 documents, both contain "data" → IDF("data") = log10(2/2) = **0**  
Only doc 1 contains "important" → IDF("important") = log10(2/1) ≈ **0.30**

In [1]:
import pandas as pd
import numpy as np

# ── Corpus ────────────────────────────────────────────────
corpus = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses"
]

# ── Step 1: Build vocabulary (unique words across all docs) ─
vocab = set()
for doc in corpus:
    vocab.update(doc.split())

vocab = sorted(vocab)                    # sorted for consistent column order
n_docs = len(corpus)
n_words = len(vocab)

print(f"Corpus size : {n_docs} documents")
print(f"Vocabulary  : {n_words} unique words")
print(f"Words       : {vocab}")


Corpus size : 2 documents
Vocabulary  : 12 unique words
Words       : ['best', 'courses', 'data', 'fields', 'important', 'is', 'most', 'of', 'one', 'science', 'the', 'this']


In [2]:
# ── Step 2: Compute Term Frequency (TF) ──────────────────
tf_matrix = pd.DataFrame(
    np.zeros((n_docs, n_words)),
    columns=vocab,
    index=[f"doc_{i}" for i in range(n_docs)]
)

for i, doc in enumerate(corpus):
    words = doc.split()
    total = len(words)
    for word in words:
        if word in tf_matrix.columns:
            tf_matrix.loc[f"doc_{i}", word] += 1 / total

print("TF Matrix (values show word frequency relative to document length):")
print(tf_matrix.round(3).to_string())


TF Matrix (values show word frequency relative to document length):
        best  courses   data  fields  important     is   most     of    one  science    the   this
doc_0  0.000    0.000  0.091   0.091      0.091  0.091  0.091  0.182  0.091    0.182  0.091  0.000
doc_1  0.111    0.111  0.111   0.000      0.000  0.111  0.000  0.111  0.111    0.111  0.111  0.111


In [3]:
# ── Step 3: Compute IDF ──────────────────────────────────
idf = {}

for word in vocab:
    doc_count = sum(1 for doc in corpus if word in doc.split())
    idf[word] = np.log10(n_docs / doc_count)

idf_series = pd.Series(idf, name="IDF").sort_values(ascending=False)

print("IDF Scores:")
print(idf_series.round(3).to_string())
print()
print("Note: IDF=0 means the word appears in EVERY document → not distinctive")


IDF Scores:
best         0.301
courses      0.301
fields       0.301
important    0.301
this         0.301
most         0.301
data         0.000
is           0.000
of           0.000
one          0.000
science      0.000
the          0.000

Note: IDF=0 means the word appears in EVERY document → not distinctive


In [4]:
# ── Step 4: Compute TF-IDF = TF × IDF ───────────────────
tfidf_matrix = tf_matrix.copy()

for word in vocab:
    tfidf_matrix[word] = tf_matrix[word] * idf[word]

print("TF-IDF Matrix:")
print(tfidf_matrix.round(4).to_string())
print()

# ── Step 5: Top keywords per document ─────────────────────
print("Top 3 keywords per document:")
for doc_id in tfidf_matrix.index:
    top = tfidf_matrix.loc[doc_id].nlargest(3)
    print(f"  {doc_id}: {list(top.index)}")


TF-IDF Matrix:
         best  courses  data  fields  important   is    most   of  one  science  the    this
doc_0  0.0000   0.0000   0.0  0.0274     0.0274  0.0  0.0274  0.0  0.0      0.0  0.0  0.0000
doc_1  0.0334   0.0334   0.0  0.0000     0.0000  0.0  0.0000  0.0  0.0      0.0  0.0  0.0334

Top 3 keywords per document:
  doc_0: ['fields', 'important', 'most']
  doc_1: ['best', 'courses', 'this']


## Key Takeaways

| Word | TF-IDF | Why |
|------|--------|-----|
| "data" | 0 | appears in both docs → IDF = 0 |
| "important" | > 0 | only in doc_0 → distinguishing |
| "courses" | > 0 | only in doc_1 → distinguishing |

## What to Try Next
- Add a 3rd document on a completely different topic — watch how IDF changes for shared words
- Use `sklearn.feature_extraction.text.TfidfVectorizer` and compare to your manual results
- Build a simple **keyword extractor**: given 5 articles, rank the top 10 important words per article